In [2]:
%load_ext dotenv
%dotenv

In [10]:
from os import getenv

from langchain.chat_models import init_chat_model
from langchain_core.rate_limiters import InMemoryRateLimiter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [6]:
or_key = getenv("OPENROUTER_KEY")

In [9]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

In [12]:
model_name = "nousresearch/hermes-3-llama-3.1-405b:free"
llm = model = init_chat_model(
    model=model_name,
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key=or_key,
)

parser = StrOutputParser()

map_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are summarizing a creative roleplay/narrative session for future continuation.\n"
     "You will be given:\n"
     "A) Prompt context (what the user asked for)\n"
     "B) The character output (the story content)\n\n"
     "IMPORTANT:\n"
     "- Use the prompt context ONLY to interpret intent.\n"
     "- Do NOT summarize or include the prompt text as part of the memory.\n"
     "- Do NOT add turns info in the resume (like instructions of how to end the turns)\n"
     "- Summarize ONLY the narrative/story facts, character state, events, plans, relationships, tone.\n"
     "- Preserve names, places, unresolved threads, and decisions.\n"
    ),
    ("user",
     "PROMPT CONTEXT (do not summarize):\n{prompt_context}\n\n"
     "MIAH OUTPUT (summarize this):\n{miah_text}\n\n"
     "Write a memory chunk with sections:\n"
     "1) What happened (dense but readable)\n"
     "2) Character state (emotions, motivations, secrets)\n"
     "3) Open threads / hooks for next time\n"
     "4) Canon facts introduced (bullet list)\n"
    )
])

map_chain = map_prompt | llm | parser


In [13]:
from langchain_core.runnables.retry import RunnableRetry

map_chain_retry = map_chain.with_retry(
    retry_if_exception_type=(Exception,),
    stop_after_attempt=3,
    wait_exponential_jitter=True,
    exponential_jitter_params={"initial": 2},
)